In [11]:
lda = LDA(2)


TypeError: LDA.__init__() takes 1 positional argument but 2 were given

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets


In [19]:
class LDA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.linear_discriminants = None # Store the eigenvector computed

    def fit(self, X, y):
        n_features = X.shape[1] # 150, 4 (iris dataset)
        class_labels = np.unique(y)

        # Calculate S_W, S_B
        mean_overall = np.mean(X, axis=0)
        S_W = np.zeros((n_features, n_features)) # 4, 4
        S_B = np.zeros((n_features, n_features)) # 4, 4

        for c in class_labels:
            X_c = X[y == c]
            mean_c = np.mean(X_c, axis=0)

            # (4, n_c) * (n_c, 4) = (4, 4)
            S_W += (X_c - mean_c).T.dot(X_c - mean_c)

            n_c = X_c.shape[0] # number of samples in classe C
            # (4, 1) * (1, 4) = (4, 4)
            mean_diff = (mean_c - mean_overall).reshape(n_features, 1) # (4, 1)

            S_B += n_c * (mean_diff).dot(mean_diff.T)

        A = np.linalg.inv(S_W).dot(S_B)
        eigenvalues, eigenvectors = np.linalg.eig(A)
        eigenvectors = eigenvectors.T
        idxs = np.argsort(abs(eigenvalues))[::-1]
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]

        self.linear_discriminants = eigenvectors[0:self.n_components]

    def transform(self, X):
        return np.dot(X, self.linear_discriminants.T) # Project the data into the linear discriminant vector (w) calculated


In [20]:
data = datasets.load_iris()
X = data.data
y = data.target
print(X.shape)
print(y.shape)


(150, 4)
(150,)


In [22]:
lda = LDA(2)
lda.fit(X, y)
X_projected = lda.transform(X)

print("Shape of X:", X.shape)
print("Shape of transformed X:", X_projected.shape)


Shape of X: (150, 4)
Shape of transformed X: (150, 2)


In [23]:
x1 = X_projected[:, 0]
x2 = X_projected[:, 1]


In [27]:
plt.scatter(x1, x2, c=y, edgecolor='none', alpha=0.8, cmap=plt.cm.get_cmap('viridis', 3))
plt.xlabel('Linear Discriminant 1')
plt.ylabel('Linear Discriminant 2')


/tmp/ipykernel_41075/2684720875.py:1: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  plt.scatter(x1, x2, c=y, edgecolor='none', alpha=0.8, cmap=plt.cm.get_cmap('viridis', 3))


Text(0, 0.5, 'Linear Discriminant 2')
